# 📊 Notebook 05: Dashboard-Ready Data Layer

## Product Line Profitability & Margin Performance Analysis  
### Nassau Candy Distributor

---

## 🔷 Objective

The objective of this notebook is to prepare all dashboard-ready datasets required for the Streamlit application.

This notebook focuses on:

- Product margin leaderboard
- Margin risk flags
- Margin risk product dataset
- Dependency indicators
- Dashboard KPI summary
- Cost-sales scatter dataset
- Final dashboard-ready export validation

### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


## 🔷 1. Load Processed Datasets

We load the processed analysis datasets created in previous notebooks.

In [2]:
df = pd.read_csv("../data/processed/final_feature_engineered_dataset.csv")
product_df = pd.read_csv("../data/processed/product_level_analysis.csv")
division_df = pd.read_csv("../data/processed/division_level_analysis.csv")
pareto_sales = pd.read_csv("../data/processed/pareto_sales_analysis.csv")
pareto_profit = pd.read_csv("../data/processed/pareto_profit_analysis.csv")

print("All processed datasets loaded successfully.")

print("Main dataset:", df.shape)
print("Product dataset:", product_df.shape)
print("Division dataset:", division_df.shape)
print("Pareto sales:", pareto_sales.shape)
print("Pareto profit:", pareto_profit.shape)

All processed datasets loaded successfully.
Main dataset: (10194, 24)
Product dataset: (15, 9)
Division dataset: (3, 9)
Pareto sales: (15, 10)
Pareto profit: (15, 10)


## 🔷 2. Product Margin Leaderboard

This dataset ranks products based on Gross Margin (%) and Profit per Unit.

It will support the Product Profitability Overview module in the Streamlit dashboard.

In [3]:
# Create product margin leaderboard

product_margin_leaderboard = product_df.sort_values(
    by=["Gross Margin %", "Profit per Unit"],
    ascending=False
).reset_index(drop=True)

product_margin_leaderboard.head(10)

,Product Name,Sales,Cost,Gross Profit,Units,Gross Margin %,Profit per Unit,Revenue Contribution %,Profit Contribution %
0,Everlasting Gobstopper,130.00,26.00,104.00,13,0.80,8.00,0.00,0.00
1,Hair Toffee,76.50,17.00,59.50,17,0.78,3.50,0.00,0.00
2,Wonka Bar - Nutty Crunch Surprise,"23,574.95","6,755.00","16,819.95",6755,0.71,2.49,0.17,0.18
3,Wonka Bar -Scrumdiddlyumptious,"27,874.80","8,517.30","19,357.50",7743,0.69,2.50,0.20,0.21
4,Wonka Bar - Fudge Mallows,"24,890.40","8,296.80","16,593.60",6914,0.67,2.40,0.18,0.18
5,Wonka Bar - Triple Dazzle Caramel,"28,485.00","9,874.80","18,610.20",7596,0.65,2.45,0.20,0.20
6,Wonka Bar - Milk Chocolate,"26,867.75","9,424.38","17,443.37",8267,0.65,2.11,0.19,0.19
7,Laffy Taffy,53.73,20.25,33.48,27,0.62,1.24,0.00,0.00
8,Fizzy Lifting Drinks,78.75,31.50,47.25,21,0.60,2.25,0.00,0.00
9,Wonka Gum,597.50,286.80,310.70,478,0.52,0.65,0.00,0.00


In [4]:
# Export product margin leaderboard

product_margin_leaderboard.to_csv(
    "../data/processed/product_margin_leaderboard.csv",
    index=False
)

print("Product margin leaderboard exported successfully.")

Product margin leaderboard exported successfully.


## 🔷 3. Margin Risk Flags

In this step, we create margin risk flags based on Gross Margin (%).

These flags will support the Cost vs Margin Diagnostics module and margin threshold filtering in the Streamlit dashboard.

In [5]:
# Define margin risk logic

def assign_margin_risk(margin):
    if margin >= 0.65:
        return "Healthy Margin"
    elif margin >= 0.50:
        return "Moderate Risk"
    elif margin >= 0.30:
        return "High Risk"
    else:
        return "Critical Review"

# Apply margin risk flag

df["Margin Risk Flag"] = df["Gross Margin %"].apply(assign_margin_risk)
product_df["Margin Risk Flag"] = product_df["Gross Margin %"].apply(assign_margin_risk)

print("Margin risk flags created successfully.")

Margin risk flags created successfully.


### Verify

In [6]:
df["Margin Risk Flag"].value_counts()

Margin Risk Flag
Healthy Margin     7714
Moderate Risk      2367
Critical Review      96
High Risk            17
Name: count, dtype: int64

## 🔷 4. Margin Risk Products Dataset

This dataset identifies products that fall under margin risk categories.

It will help highlight products that may require pricing review, cost optimization, or discontinuation review.

In [7]:
# Create margin risk products dataset

margin_risk_products = product_df[
    product_df["Margin Risk Flag"].isin(["High Risk", "Critical Review"])
].sort_values(by="Gross Margin %", ascending=True)

margin_risk_products

,Product Name,Sales,Cost,Gross Profit,Units,Gross Margin %,Profit per Unit,Revenue Contribution %,Profit Contribution %,Margin Risk Flag
4,Kazookles,"1,205.75","1,113.00",92.75,371,0.08,0.25,0.01,0.00,Critical Review
2,Fun Dip,12.00,7.20,4.80,8,0.40,0.60,0.00,0.00,High Risk
7,Nerds,15.00,8.00,7.00,10,0.47,0.70,0.00,0.00,High Risk
8,SweeTARTS,61.50,32.80,28.70,41,0.47,0.70,0.00,0.00,High Risk


In [8]:
# Export margin risk products dataset

margin_risk_products.to_csv(
    "../data/processed/margin_risk_products.csv",
    index=False
)

print("Margin risk products exported successfully.")

Margin risk products exported successfully.


## 🔷 5. Dependency Indicators

This section creates summary indicators to measure revenue and profit concentration risk.

These indicators will support the Profit Concentration Analysis module in the Streamlit dashboard.

In [9]:
# Calculate dependency indicators

products_80_revenue = pareto_sales[pareto_sales["Cumulative Sales %"] <= 0.80].shape[0]
products_80_profit = pareto_profit[pareto_profit["Cumulative Profit %"] <= 0.80].shape[0]

total_products = product_df["Product Name"].nunique()

dependency_indicators = pd.DataFrame({
    "Metric": [
        "Products contributing to 80% revenue",
        "Products contributing to 80% profit",
        "Total products",
        "Revenue dependency ratio",
        "Profit dependency ratio"
    ],
    "Value": [
        products_80_revenue,
        products_80_profit,
        total_products,
        products_80_revenue / total_products,
        products_80_profit / total_products
    ]
})

dependency_indicators

,Metric,Value
0,Products contributing to 80% revenue,4.00
1,Products contributing to 80% profit,4.00
2,Total products,15.00
3,Revenue dependency ratio,0.27
4,Profit dependency ratio,0.27


In [10]:
# Export dependency indicators

dependency_indicators.to_csv(
    "../data/processed/dependency_indicators.csv",
    index=False
)

print("Dependency indicators exported successfully.")

Dependency indicators exported successfully.


## 🔷 6. Dashboard KPI Summary

This dataset contains key summary metrics required for the dashboard KPI cards.

It includes:

- Total Sales
- Total Profit
- Total Units
- Average Margin
- Margin Volatility

In [11]:
# Calculate KPI summary

total_sales = df["Sales"].sum()
total_profit = df["Gross Profit"].sum()
total_units = df["Units"].sum()
avg_margin = df["Gross Margin %"].mean()

# Recalculate margin volatility (monthly)

df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Order Month"] = df["Order Date"].dt.to_period("M")

monthly_df = df.groupby("Order Month").agg({
    "Sales": "sum",
    "Gross Profit": "sum"
}).reset_index()

monthly_df["Monthly Margin"] = monthly_df["Gross Profit"] / monthly_df["Sales"]

margin_volatility = monthly_df["Monthly Margin"].std()

# Create KPI summary dataframe

kpi_summary = pd.DataFrame({
    "Metric": [
        "Total Sales",
        "Total Profit",
        "Total Units",
        "Average Margin",
        "Margin Volatility"
    ],
    "Value": [
        total_sales,
        total_profit,
        total_units,
        avg_margin,
        margin_volatility
    ]
})

kpi_summary

,Metric,Value
0,Total Sales,"141,783.63"
1,Total Profit,"93,442.80"
2,Total Units,"38,654.00"
3,Average Margin,0.67
4,Margin Volatility,0.01


In [12]:
# Export KPI summary

kpi_summary.to_csv(
    "../data/processed/dashboard_kpi_summary.csv",
    index=False
)

print("Dashboard KPI summary exported successfully.")

Dashboard KPI summary exported successfully.


## 🔷 7. Cost-Sales Scatter Dataset

This dataset prepares product-level values for cost-sales scatter visualization.

It includes:

- Product Name
- Sales
- Cost
- Gross Profit
- Gross Margin %
- Cost Ratio
- Margin Risk Flag

In [13]:
# Create cost-sales scatter dataset

cost_sales_scatter = product_df.copy()

cost_sales_scatter["Cost Ratio"] = cost_sales_scatter["Cost"] / cost_sales_scatter["Sales"]

cost_sales_scatter = cost_sales_scatter[[
    "Product Name",
    "Sales",
    "Cost",
    "Gross Profit",
    "Gross Margin %",
    "Profit per Unit",
    "Cost Ratio",
    "Margin Risk Flag"
]].sort_values(by="Cost Ratio", ascending=False)

cost_sales_scatter.head()

,Product Name,Sales,Cost,Gross Profit,Gross Margin %,Profit per Unit,Cost Ratio,Margin Risk Flag
4,Kazookles,"1,205.75","1,113.00",92.75,0.08,0.25,0.92,Critical Review
2,Fun Dip,12.00,7.20,4.80,0.40,0.60,0.60,High Risk
7,Nerds,15.00,8.00,7.00,0.47,0.70,0.53,High Risk
8,SweeTARTS,61.50,32.80,28.70,0.47,0.70,0.53,High Risk
6,Lickable Wallpaper,"7,860.00","3,930.00","3,930.00",0.50,10.00,0.50,Moderate Risk


In [14]:
# Export cost-sales scatter dataset

cost_sales_scatter.to_csv(
    "../data/processed/cost_sales_scatter_dataset.csv",
    index=False
)

print("Cost-sales scatter dataset exported successfully.")

Cost-sales scatter dataset exported successfully.


## 🔷 8. Final Dashboard-Ready Export Validation

In this step, we validate whether all required dashboard-ready CSV files have been successfully created inside the `data/processed` folder.

In [15]:
import os

required_files = [
    "final_feature_engineered_dataset.csv",
    "product_level_analysis.csv",
    "division_level_analysis.csv",
    "product_margin_leaderboard.csv",
    "margin_risk_products.csv",
    "dependency_indicators.csv",
    "dashboard_kpi_summary.csv",
    "cost_sales_scatter_dataset.csv",
    "pareto_sales_analysis.csv",
    "pareto_profit_analysis.csv"
]

processed_path = "../data/processed"

validation_results = []

for file in required_files:
    file_path = os.path.join(processed_path, file)
    validation_results.append({
        "File Name": file,
        "Exists": os.path.exists(file_path)
    })

validation_df = pd.DataFrame(validation_results)

validation_df

,File Name,Exists
0,final_feature_engineered_dataset.csv,True
1,product_level_analysis.csv,True
2,division_level_analysis.csv,True
3,product_margin_leaderboard.csv,True
4,margin_risk_products.csv,True
5,dependency_indicators.csv,True
6,dashboard_kpi_summary.csv,True
7,cost_sales_scatter_dataset.csv,True
8,pareto_sales_analysis.csv,True
9,pareto_profit_analysis.csv,True
